# UNData API Exercise

In this exercise, you'll redo the data gathering phase of the UNData Exploration project by using APIs instead of downloading csv files.

You'll make use of the [World Bank Indicators API](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392-about-the-indicators-api-documentation). Note that this API does not require an API key. Before attempting the exercise, it would be a good idea to skim through the Documentation page and to check out the [Basic Call Structure article](https://datahelpdesk.worldbank.org/knowledgebase/articles/898581).

## Install dependencies

## Import dependencies

In [1]:
import pandas as pd
import requests
import io
import re
import time
from bs4 import BeautifulSoup as bs

## Exercises

#### Globals

In [2]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

v2_url = "https://api.worldbank.org/v2"
indicator_api = "indicator"
country_api = "country"

#### 1. Use the API to get all available data for the _GDP per capita, PPP (constant 2017 international $)_ indicator. Hint: this indicator has code "NY.GDP.PCAP.PP.KD". Adjust the query parameters so that you can retrieve all available rows. Convert the results to a DataFrame.

In [3]:
def e1():
    return get_data(indicator_api, resource="NY.GDP.PCAP.PP.KD", params=f"per_page={30000}")

def get_url(api, resource="", params=""):
    apis = {"indicator": "country/all/indicator", "country": "country"}
    if (api in apis):
        return f"{v2_url}/{apis[api]}/{resource}?{params}"
    else:
        raise ValueError(f"Invalid api: '{api}'! This client only supports the following APIs: {apis}.")
    
def get_data(api, resource="", params=""):
    start_time = time.time()
    
    # Get initial page data and total page count
    url = get_url(api, resource, params)
    response = requests.get(f"{url}{'' if url.endswith('?') else '&'}format=json&page=1", headers=headers).json()
    total_pages = response[0]["pages"]
    all_data = []
    all_data.extend(response[1])
    page_time = time.time() - start_time
    print(f"Fetched and processed {len(all_data)} items from page 1 of {total_pages} total pages in {page_time} seconds.")
    print(f"Estimated completion time: {page_time * total_pages} seconds ({page_time * (total_pages - 1)} seconds from now).")
    
    # Iterate through all remaining pages
    for page in range(2, total_pages + 1):
        print(f"Fetching page {page} for processing...")
        page_url = f"{url}&page={page}"
        response = requests.get(page_url).json()
        all_data.extend(response[1]) # Data is always in the second element of the list
    
    print(f"Processed {len(all_data)} records across {total_pages} pages in {time.time() - start_time} seconds.")
    return pd.DataFrame(all_data)
    
e1 = e1()
e1

Fetched and processed 17556 items from page 1 of 1 total pages in 0.7457947731018066 seconds.
Estimated completion time: 0.7457947731018066 seconds (0.0 seconds from now).
Processed 17556 records across 1 pages in 0.7470693588256836 seconds.


,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"{'id': 'NY.GDP.PCAP.PP.KD', 'value': 'GDP per ...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2025,NaN,,,0
1,"{'id': 'NY.GDP.PCAP.PP.KD', 'value': 'GDP per ...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2024,4106.434140,,,0
2,"{'id': 'NY.GDP.PCAP.PP.KD', 'value': 'GDP per ...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2023,4083.993003,,,0
3,"{'id': 'NY.GDP.PCAP.PP.KD', 'value': 'GDP per ...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2022,4106.501907,,,0
4,"{'id': 'NY.GDP.PCAP.PP.KD', 'value': 'GDP per ...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2021,4056.113017,,,0
...,...,...,...,...,...,...,...,...
17551,"{'id': 'NY.GDP.PCAP.PP.KD', 'value': 'GDP per ...","{'id': 'ZW', 'value': 'Zimbabwe'}",ZWE,1964,NaN,,,0
17552,"{'id': 'NY.GDP.PCAP.PP.KD', 'value': 'GDP per ...","{'id': 'ZW', 'value': 'Zimbabwe'}",ZWE,1963,NaN,,,0
17553,"{'id': 'NY.GDP.PCAP.PP.KD', 'value': 'GDP per ...","{'id': 'ZW', 'value': 'Zimbabwe'}",ZWE,1962,NaN,,,0
17554,"{'id': 'NY.GDP.PCAP.PP.KD', 'value': 'GDP per ...","{'id': 'ZW', 'value': 'Zimbabwe'}",ZWE,1961,NaN,,,0


In [4]:
# def e1b():
#     return [e.text for e in base_soup.find_all("h2", attrs={"class": "title is-5"})]
# e1b = e1b()
# e1b

#### 2. Now, use the API to get all available data for _Life expectancy at birth, total (years)_. This indicator has code "SP.DYN.LE00.IN". Again, convert the results to a DataFrame.

In [5]:
def e2():
    return get_data(indicator_api, resource="SP.DYN.LE00.IN", params=f"per_page={30000}")
e2 = e2()
e2

Fetched and processed 17556 items from page 1 of 1 total pages in 0.4919872283935547 seconds.
Estimated completion time: 0.4919872283935547 seconds (0.0 seconds from now).
Processed 17556 records across 1 pages in 0.49254584312438965 seconds.


,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"{'id': 'SP.DYN.LE00.IN', 'value': 'Life expect...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2025,NaN,,,0
1,"{'id': 'SP.DYN.LE00.IN', 'value': 'Life expect...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2024,65.349930,,,0
2,"{'id': 'SP.DYN.LE00.IN', 'value': 'Life expect...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2023,65.146228,,,0
3,"{'id': 'SP.DYN.LE00.IN', 'value': 'Life expect...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2022,64.487152,,,0
4,"{'id': 'SP.DYN.LE00.IN', 'value': 'Life expect...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2021,62.979999,,,0
...,...,...,...,...,...,...,...,...
17551,"{'id': 'SP.DYN.LE00.IN', 'value': 'Life expect...","{'id': 'ZW', 'value': 'Zimbabwe'}",ZWE,1964,55.431000,,,0
17552,"{'id': 'SP.DYN.LE00.IN', 'value': 'Life expect...","{'id': 'ZW', 'value': 'Zimbabwe'}",ZWE,1963,54.942000,,,0
17553,"{'id': 'SP.DYN.LE00.IN', 'value': 'Life expect...","{'id': 'ZW', 'value': 'Zimbabwe'}",ZWE,1962,54.453000,,,0
17554,"{'id': 'SP.DYN.LE00.IN', 'value': 'Life expect...","{'id': 'ZW', 'value': 'Zimbabwe'}",ZWE,1961,53.966000,,,0


#### 3. Merge the two results DataFrames together. You may want to rename or drop columns prior to merging.

In [6]:
def e3():
    columns = ["countryiso3code", "date", "value"]
    gdp = e1[columns].rename(columns={"countryiso3code": "id", "date": "year", "value": "gdp"}, inplace=False)
    leb = e2[columns].rename(columns={"countryiso3code": "id", "date": "year", "value": "leb"}, inplace=False)
    return pd.merge(
        gdp,
        leb,
        on=["id", "year"],
        how="inner"
    )

e3 = e3()
e3

,id,year,gdp,leb
0,AFE,2025,NaN,NaN
1,AFE,2024,4106.434140,65.349930
2,AFE,2023,4083.993003,65.146228
3,AFE,2022,4106.501907,64.487152
4,AFE,2021,4056.113017,62.979999
...,...,...,...,...
18871,ZWE,1964,NaN,55.431000
18872,ZWE,1963,NaN,54.942000
18873,ZWE,1962,NaN,54.453000
18874,ZWE,1961,NaN,53.966000


#### 4. You can also get more information about the available countries (region, capital city, income level classification, etc.) by using the [Country API](https://datahelpdesk.worldbank.org/knowledgebase/articles/898590-country-api-queries). Use this API to pull in all available data. Merge this with your other datasets. Use this to now remove the rows that correspond to regions and not countries.

In [7]:
def e4():
    df = get_data(country_api, params=f"per_page={300}")
    mask = df["region"].apply(lambda region: region["value"] != "Aggregates") # while most regions have a digit in their ISO2 not all do but all regions do have "Aggregates" as their region.value
    countries = df[mask].reset_index(drop=True)
    countries["country"] = countries["name"]
    print(f"Found {len(countries)} countries to merge...")
    return pd.merge(
        countries,
        e3,
        on="id",
        how="left"
    )[["year", "country", "gdp", "leb"]].sort_values(by=["country", "year"])

e4 = e4()
e4

Fetched and processed 296 items from page 1 of 1 total pages in 0.2986774444580078 seconds.
Estimated completion time: 0.2986774444580078 seconds (0.0 seconds from now).
Processed 296 records across 1 pages in 0.29900622367858887 seconds.
Found 217 countries to merge...


,year,country,gdp,leb
131,1960,Afghanistan,NaN,32.799
130,1961,Afghanistan,NaN,33.291
129,1962,Afghanistan,NaN,33.757
128,1963,Afghanistan,NaN,34.201
127,1964,Afghanistan,NaN,34.673
...,...,...,...,...
14260,2021,Zimbabwe,4827.088694,60.135
14259,2022,Zimbabwe,5036.761361,62.360
14258,2023,Zimbabwe,5218.022665,62.775
14257,2024,Zimbabwe,5215.253011,63.064
